In [1]:
!pip install langchain langchain_community langchain_google_genai langchain_classic langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
import os

from google.colab import userdata
gem = userdata.get('gemini')

os.environ["GOOGLE_API_KEY"] = gem

In [3]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import LLMChain

In [4]:
# ConversationBufferWindowMemory

from langchain_classic.memory import ConversationBufferWindowMemory

In [5]:
memory1 = ConversationBufferWindowMemory(memory_key="previous_chat", k=2,return_messages=True) # if not give this key it will take "History" as a default key name

# ConversationBufferWindowMemory it create a list of dictionary inside that dictionary the entire thing will be stored as again in list to that list we are giving a key and that key is a memory_key
# Which will be used to put inside chat prompt template in your Messagesplaceholder

# [{"previous_chat": [store all messages]}]

# Whenever we are uisng conversation buffer window memory this k is going to store a pair of your [human message and ai message] together it will take as a one conversation and it will not store system message.
# most IMP: return_messages=True it will return list of message. if it is Falso means it will be returning string as an output and that the reason why it will be not able to work with chat prompt template.

/tmp/ipykernel_3418/3068922929.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory1 = ConversationBufferWindowMemory(memory_key="previous_chat", k=2,return_messages=True) # if not give this key it will take "History" as a default key name


In [6]:
# add dummy data

memory1.save_context( {"input":"hi"} , {"output":"hi how are you"})
memory1.save_context( {"input":"bye"} , {"output":"bye"})
memory1.save_context( {"input":"sdaa"} , {"output":"hscc"})

In [7]:
memory1.load_memory_variables({}) #K=2

{'previous_chat': [HumanMessage(content='bye', additional_kwargs={}, response_metadata={}),
  AIMessage(content='bye', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='sdaa', additional_kwargs={}, response_metadata={}),
  AIMessage(content='hscc', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [8]:
#Create a ChatpromptTemplate (cpt)

cpt = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("You are a helpful assistance"),
                                        MessagesPlaceholder(variable_name = "previous_chat"),
                                        HumanMessagePromptTemplate.from_template("{input}")])

# MessagesPlaceholder: It is also prompt template by using which we can insert a list of message inside your message (ChatPromptTemplate.from_messages).


model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
sto = StrOutputParser()

In [9]:
chain_memory=LLMChain(prompt = cpt ,
                llm = model,
                output_parser = sto ,
                memory = memory1)

/tmp/ipykernel_3418/3165600647.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain_memory=LLMChain(prompt = cpt ,


In [10]:
chain_memory.run({"input":"Hi"})

# First it called to memory : It will be empty


/tmp/ipykernel_3418/894721975.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  chain_memory.run({"input":"Hi"})


'Hello! How can I help you today?'

In [11]:
chain_memory.invoke({"input":"Hi"})["previous_chat"][-1].content

'Hello! How can I help you today?'

In [12]:
while True:

  query = input("Type the user prompt")

  print("Human message: {}".format(query))

  aimes = chain_memory.invoke({"input":query})["previous_chat"][-1].content

  print("ai message : {}".format(aimes))

  if query == "bye":
    data = input("Type save or clear the memory")

    if data.lower() == "clear":
      memory1.clear()
    break

Type the user prompthi
Human message: hi
ai message : Hello again! How can I help you today?
Type the user prompti am shrutika
Human message: i am shrutika
ai message : Hi there! How can I help you today?
Type the user prompti am ai ml developer
Human message: i am ai ml developer
ai message : Nice to meet you, Shrutika!

How can I help you today? Is there something you'd like to talk about or something I can assist you with?
Type the user promptgive my introduction in impressive way
Human message: give my introduction in impressive way
ai message : That's fantastic, Shrutika! That's a really exciting and rapidly evolving field.

What kind of AI/ML development are you typically involved in? Are you working on specific applications, research, or something else?
Type the user promptbye
Human message: bye
ai message : Okay, Shrutika, let's craft an introduction that truly highlights your expertise and passion as an AI/ML Developer. The best one will depend on the *context* (who you're tal

In [13]:
memory1.load_memory_variables({})

{'previous_chat': [HumanMessage(content='give my introduction in impressive way', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Okay, Shrutika, let\'s craft an introduction that truly highlights your expertise and passion as an AI/ML Developer. The best one will depend on the *context* (who you\'re talking to, what the situation is), but here are a few impressive options, ranging from visionary to more technical:\n\n---\n\n**Option 1: The Visionary & Impactful**\n\n"Hello, I\'m Shrutika, and I\'m an AI/ML Developer. I don\'t just write code; I engineer intelligence. My passion lies in transforming raw data into the intelligent systems and predictive models that are shaping the future, solving complex problems, and creating groundbreaking opportunities across industries."\n\n---\n\n**Option 2: The Problem-Solver & Innovator**\n\n"Nice to meet you, I\'m Shrutika. As an AI/ML Developer, I thrive on the challenge of taking intricate data and turning it into elegant, hig

In [14]:
# All the above practice for that we used chain(LLMChain) not runnables

#### How to mimic the same thing using Runnables

- For this we need 3 things

1. Chain[Runnables]: E.g: SequencalRunnable

2. Session_store : It is a variable which is dtore all conversation with respect to their particular session_id and we can store this Locally (store in dictionary) or cloud.

3. Major_part: Session Factory : It will be store 2 things : It will create a store and inside the session store be going on adding memory.

[session factory(session_store)] : This session_Store will be connected and accessed by session_factory: Whenever you require that particular session history it will give you the session

- store the data and load the data


### Note: we wrap this all 3 steps and inclosed in RunnableMessageHistory and it's duty is to wrap this 3 thing together do communication between chain,session_store and session_factory

In [15]:
from langchain_core.messages import trim_messages, HumanMessage, AIMessage, SystemMessage

# trim_messages : With the help of this function we can able to create a function by using which we can trip the messages

In [16]:
trim = trim_messages(max_tokens = 4,
              token_counter=len ,
              include_system=True,
              strategy="last",
              start_on="human")

# max_tokens: maximum value that can be allowed it is same as  k=2
# max_token and token_counte are work together
# token_counter is acting as a function and return value , and that value
# token_counter value compare with the max_token value
# token_counter=len : each message acting as a one unit
# Note: This will return Runnablelambda means your trim_message is going to return you a function and this function is a runnable object which i can use inside the chain.

In [20]:
# create dummy data
message = [SystemMessage(content = "You are a assisteant"),
           HumanMessage(content = "Hello"),
           AIMessage(content = "Hi, How are you"),
           HumanMessage(content="my name is p1"),
           AIMessage(content="Nice to meet you p1")]

message

[SystemMessage(content='You are a assisteant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi, How are you', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='my name is p1', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice to meet you p1', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [18]:
trim.invoke(message)

[SystemMessage(content='You are a assisteant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='my name is p1', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice to meet you p1', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [19]:
#How to count no of tokens for ChatGPT model

In [22]:
import tiktoken
# It is a tokenisation library developed by OpenAI
# use : count the tokens

# I just write a syntax how to work with Openai, So use a valid OpenAI API to try the below code.

In [25]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.2 MB/s eta 0:00:00


In [26]:
from langchain_openai import ChatOpenAI

In [ ]:
import os
from google.colab import userdata
oi = userdata.get('openAI')

os.environ["OPENAI_API_KEY"] =oi

In [ ]:
model = ChatOpenAI(model="gpt-5.2-2025-12-11")

In [ ]:
tik =tiktoken.get_encoding("cl100k_base") # used tokenization encoding

#### cl100k_base: it is a widely used tokenization encoding developed by OpenAI, designed to convert text into tokens (numerical representations) that LLMs can process. It is the standard tokenizer for popular models including GPT-4, GPT-3.5-turbo, and text-embedding-ada-002.

In [ ]:
len(tik.encode("shrutika"))

In [ ]:
tik.decode([82])

In [ ]:
# Real world usage
trim = trim_messages(max_tokens=10,
                     token_counter=model,
                     include_system =True,
                     strategy ="last",
                     start_on="human")

# here we count the number of tokens and based on the tokens tell me the answer

In [ ]:
trim.invoke(message)

In [ ]:
for y in trim.invoke(message):
  print(len(tik.encode(y.content)))

In [29]:
# Now try with google_gemini

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [30]:
model.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d70f7-3cb2-7703-825a-15a28be661e8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 82, 'total_tokens': 84, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 73}})

In [ ]:
#Count how many tokens  it is going to use

In [31]:
model.get_num_tokens("hi")

1

In [32]:
# Real world usage
trim = trim_messages(max_tokens=40,
                     token_counter=model,
                     include_system =True,
                     strategy ="last",
                     start_on="human")

In [33]:
message

[SystemMessage(content='You are a assisteant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi, How are you', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='my name is p1', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice to meet you p1', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [34]:
def count_tokens(message):
  data = "\n".join([y.content for y in message])
  return model.get_num_tokens(data)

In [41]:
#Production level
trim = trim_messages(max_tokens=40,
                     token_counter=count_tokens,
                     include_system =True,
                     strategy ="last",
                     start_on="human")

In [42]:
trim.invoke(message)

[SystemMessage(content='You are a assisteant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi, How are you', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='my name is p1', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice to meet you p1', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

### Create ConversationalBufferWindowMemory

In [43]:
from langchain_core.runnables.history import RunnableWithMessageHistory # Wrapper
from langchain_core.chat_history import InMemoryChatMessageHistory #RedisChatMessageHistory #All the different type of session_store

# InMemoryChatMessageHistory: It is called local memory, when the sessional memory is over all the chat will be losed
# RedisChatMessageHistory: It is a CLOUD PLATFORM by using which we can create a cloud database
# Redis is a platform : Data stored in RAM → microsecond-level latency
# Much faster than disk-based DBs like MySQL
# Unlike traditional databases, Redis stores data in RAM, making it extremely fast

In [44]:
# Create a session_store

storage ={}

def session_factory(session_id):
  if session_id not in storage:
   storage[session_id] = InMemoryChatMessageHistory()
  return storage[session_id]

In [45]:
trim = trim_messages(max_tokens=4,
                     token_counter=len,
                     include_system =True,
                     strategy ="last",
                     start_on="human")

In [46]:
#Create a ChatpromptTemplate (cpt)

cpt = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("You are a helpful assistance"),
                                        MessagesPlaceholder(variable_name = "previous_chat"),
                                        HumanMessagePromptTemplate.from_template("{input}")])

# MessagesPlaceholder: It is also prompt template by using which we can insert a list of message inside your message (ChatPromptTemplate.from_messages).


model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
sto = StrOutputParser()

In [62]:
chain = cpt|trim|model

In [63]:
chain_with_memory = RunnableWithMessageHistory(runnable=chain,
                                               get_session_history=session_factory,
                                               history_messages_key="previous_chat",
                                               input_messages_key="input")

In [64]:
chain_with_memory.invoke({"input":"my name is shrutika"},config = {"configurable":{"session_id":"person2"}}) #It is not able to answer

AIMessage(content='Your name is Shrutika.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d7112-2239-7b33-95e0-8733d50b9827-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 32, 'total_tokens': 56, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 25}})

In [65]:
chain_with_memory.invoke({"input":"What is my name"},config = {"configurable":{"session_id":"person2"}}) #It is not able to answer

AIMessage(content='Your name is **Shrutika**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d7112-2aac-7453-9bc8-d5a6673331cb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 40, 'total_tokens': 65, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 32}})